# Youtube Top 1000 By Subscribers

A short, personalized walkthrough built from this dataset's actual values.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but those won't be saved outside of the current session


## 1. Load and Inspect

Read the data and check shape, sample rows, and missingness.

In [ ]:
from pathlib import Path

csv_name = 'youtube_top_1000_by_subscribers.csv'
candidate_paths = list(Path('/kaggle/input').rglob(csv_name))
path = str(candidate_paths[0]) if candidate_paths else csv_name

df = pd.read_csv(path)
print('Loaded from:', path)
print('shape:', df.shape)
display(df.head())

profile = pd.DataFrame({
    'missing': df.isna().sum(),
    'dtype': df.dtypes.astype(str),
})
display(profile)


## 2. Dataset Snapshot

These quick notes are derived from the current dataset output:

- The dataset has **1,000 rows** and **11 columns**.
- Top `title` by `subscribers` is **MrBeast** (476,000,000).
- `subscribers` median is **24,800,000** with a maximum of **476,000,000**.
- Missing values exist (total **1077**), mainly in `default_language` (862), `country` (146), `description` (56).

## 3. Visual Summary

A compact visual pass on leaders and metric distribution.

In [ ]:
def to_numeric_human(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    cleaned = series.astype(str).str.strip().str.replace(',', '', regex=False)
    mult = cleaned.str.extract(r'(?i)([KMB])$')[0].str.upper().map({'K':1e3, 'M':1e6, 'B':1e9}).fillna(1)
    nums = pd.to_numeric(cleaned.str.replace(r'(?i)[KMB]$', '', regex=True), errors='coerce')
    return nums * mult

entity_candidates = ['channel','Channel Name','account','username','page','name','title','repo-name','student_id']
metric_candidates = ['subscribers','followers','likes','views','video_views','stars','final_score','forks','talking_about','uploads','videos','open-issues']

entity_col = next((c for c in entity_candidates if c in df.columns), None)
metric_col = next((c for c in metric_candidates if c in df.columns), None)

if metric_col is None:
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c.lower() != 'rank']
    metric_col = numeric_cols[0] if numeric_cols else None

metric_num = to_numeric_human(df[metric_col]) if metric_col else pd.Series(dtype=float)

secondary_col = None
for c in df.columns:
    if c == metric_col or c.lower() == 'rank':
        continue
    candidate = to_numeric_human(df[c])
    if candidate.notna().sum() >= max(5, int(0.5 * len(df))):
        secondary_col = c
        secondary_num = candidate
        break

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if entity_col and metric_col and metric_num.notna().any():
    top_plot = pd.DataFrame({entity_col: df[entity_col], '__metric_num': metric_num}).dropna().sort_values('__metric_num', ascending=False).head(10).sort_values('__metric_num')
    sns.barplot(data=top_plot, y=entity_col, x='__metric_num', hue=entity_col, dodge=False, legend=False, ax=axes[0], palette='crest')
    axes[0].set_title(f'Top 10 by {metric_col}')
    axes[0].set_xlabel(f'{metric_col} (numeric)')
    axes[0].set_ylabel('')
else:
    axes[0].text(0.5, 0.5, 'No suitable entity/metric pair found', ha='center', va='center')
    axes[0].set_axis_off()

if metric_col and metric_num.notna().any():
    sns.histplot(metric_num.dropna(), bins=20, kde=True, ax=axes[1], color='#2a9d8f')
    axes[1].set_title(f'Distribution of {metric_col}')
    axes[1].set_xlabel(f'{metric_col} (numeric)')
else:
    axes[1].text(0.5, 0.5, 'No numeric metric found', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

if metric_col and secondary_col and metric_num.notna().any():
    secondary_num = to_numeric_human(df[secondary_col])
    scatter_df = pd.DataFrame({'x': secondary_num, 'y': metric_num}).dropna()
    if len(scatter_df) >= 10:
        plt.figure(figsize=(7, 5))
        sns.scatterplot(data=scatter_df, x='x', y='y', s=70, alpha=0.75, color='#264653')
        plt.title(f'{metric_col} vs {secondary_col}')
        plt.xlabel(f'{secondary_col} (numeric)')
        plt.ylabel(f'{metric_col} (numeric)')
        plt.tight_layout()
        plt.show()


## 4. Key Takeaways

- Use the top-10 chart to identify dominant entities quickly.
- Use the distribution chart to judge spread and skew in the main metric.
- Use the scatter plot to inspect how the main metric moves with another numeric signal.